In [ ]:
!pip install keras-tuner -q
import nltk, re
from nltk.corpus import gutenberg
nltk.download('gutenberg')

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 1: Load and clean Shakespeare text
text = gutenberg.raw('shakespeare-hamlet.txt')
text = text.lower().replace('\n', ' ')
text = re.sub(r'[^a-zA-Z ]', '', text)

# Step 2: Tokenize
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

# Step 3: Create smaller input sequences for quick tuning
input_sequences = []
for line in text.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(3, len(token_list)):
        n_gram_seq = token_list[:i]
        input_sequences.append(n_gram_seq)

# Keep only first 3000 samples to avoid memory crash
input_sequences = input_sequences[:3000]

max_seq_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))
X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = keras.utils.to_categorical(y, num_classes=total_words)

print(f"✅ Using smaller dataset — Shape of X: {X.shape}, y: {y.shape}")

# Step 4: Build lightweight model
def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Embedding(total_words, hp.Choice('embedding_dim', [64, 128]), input_length=max_seq_len - 1))
    model.add(layers.LSTM(units=hp.Choice('lstm_units', [64, 128]), dropout=0.3))
    model.add(layers.Dense(total_words, activation='softmax'))
    model.compile(loss='categorical_crossentropy',
                  optimizer=keras.optimizers.Adam(learning_rate=hp.Choice('learning_rate', [1e-3, 1e-4])),
                  metrics=['accuracy'])
    return model

# Step 5: Quick tuning
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=2,    # only 2 combos
    executions_per_trial=1,
    directory='lstm_tuning_small',
    project_name='text_gen_small'
)

# Step 6: Short training for test output
tuner.search(X, y, epochs=2, batch_size=64, validation_split=0.1, verbose=1)

best_model = tuner.get_best_models(num_models=1)[0]
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n✅ Best hyperparameters found:")
for param in best_hps.values.keys():
    print(f"{param}: {best_hps.get(param)}")

# Step 7: Train final model (light)
history = best_model.fit(X, y, epochs=2, batch_size=64, validation_split=0.1)

# Step 8: Generate sample text
def generate_text(seed_text, next_words=15):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
        predicted = np.argmax(best_model.predict(token_list), axis=-1)[0]
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

print("\n📝 Example Generated Text:")
print(generate_text("to be or not to", next_words=15))


Trial 2 Complete [00h 08m 48s]
val_accuracy: 0.013333333656191826

Best val_accuracy So Far: 0.013333333656191826
Total elapsed time: 00h 13m 33s

✅ Best hyperparameters found:
embedding_dim: 64
lstm_units: 64
learning_rate: 0.0001
Epoch 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


43/43 ━━━━━━━━━━━━━━━━━━━━ 141s 3s/step - accuracy: 0.0193 - loss: 8.4498 - val_accuracy: 0.0067 - val_loss: 8.3939
Epoch 2/2
43/43 ━━━━━━━━━━━━━━━━━━━━ 145s 3s/step - accuracy: 0.0145 - loss: 8.2561 - val_accuracy: 0.0067 - val_loss: 8.0023

📝 Example Generated Text:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 352ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 168ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
to be or not to this this this this this this this this this this this this this this this


In [ ]:
!pip install keras-tuner -q
import nltk, re, numpy as np, tensorflow as tf, keras_tuner as kt
from nltk.corpus import gutenberg
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 1: Load and clean Shakespeare text
nltk.download('gutenberg')
text = gutenberg.raw('shakespeare-hamlet.txt').lower().replace('\n', ' ')
text = re.sub(r'[^a-zA-Z ]', '', text)

# Step 2: Tokenize
tokenizer = Tokenizer(num_words=3000)  # limit vocab to top 3000 words
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

# Step 3: Create input sequences
input_sequences = []
for line in text.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(3, len(token_list)):
        n_gram_seq = token_list[:i]
        input_sequences.append(n_gram_seq)

# Use first 8000 samples for better learning but still efficient
input_sequences = input_sequences[:8000]

max_seq_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))
X, y = input_sequences[:, :-1], input_sequences[:, -1]
y = keras.utils.to_categorical(y, num_classes=total_words)

print(f"✅ Dataset ready — X: {X.shape}, y: {y.shape}, vocab: {total_words}")

# Step 4: Build a better LSTM model
model = keras.Sequential([
    layers.Embedding(total_words, 128, input_length=max_seq_len - 1),
    layers.LSTM(256, return_sequences=False, dropout=0.3, recurrent_dropout=0.2),
    layers.Dense(total_words, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer=keras.optimizers.Adam(learning_rate=0.001), metrics=['accuracy'])
model.summary()

# Step 5: Train for more epochs
history = model.fit(X, y, epochs=20, batch_size=64, validation_split=0.1, verbose=1)

# Step 6: Generate better text
def generate_text(seed_text, next_words=20):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
        predicted = np.argmax(model.predict(token_list), axis=-1)[0]
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

print("\n📝 Example Generated Text:")
print(generate_text("to be or not to", next_words=20))


[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


✅ Dataset ready — X: (8000, 8001), y: (8000, 4797), vocab: 4797


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20


In [1]:
!pip install keras-tuner -q
import nltk, re, numpy as np, tensorflow as tf
from nltk.corpus import gutenberg
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Step 1: Load and clean text
nltk.download('gutenberg')
text = gutenberg.raw('shakespeare-hamlet.txt').lower().replace('\n', ' ')
text = re.sub(r'[^a-zA-Z ]', '', text)

# Step 2: Tokenization (limit vocab for memory safety)
tokenizer = Tokenizer(num_words=2000)  # reduced vocab
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

# Step 3: Create fewer input sequences
input_sequences = []
for line in text.split('.'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(3, len(token_list)):
        input_sequences.append(token_list[:i])

input_sequences = input_sequences[:4000]  # safe limit
max_seq_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))

X, y = input_sequences[:, :-1], input_sequences[:, -1]

# Use sparse labels (saves huge RAM)
print(f" Data ready: X={X.shape}, y={y.shape}, vocab={total_words}")

# Step 4: Define smaller model
model = keras.Sequential([
    layers.Embedding(total_words, 64, input_length=max_seq_len - 1),
    layers.LSTM(128, dropout=0.3, recurrent_dropout=0.2),
    layers.Dense(total_words, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',
              optimizer=keras.optimizers.Adam(learning_rate=0.001),
              metrics=['accuracy'])

# Step 5: Train (short but effective)
history = model.fit(X, y, epochs=10, batch_size=64, validation_split=0.1, verbose=1)

# Step 6: Generate text
def generate_text(seed_text, next_words=20):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
        predicted = np.argmax(model.predict(token_list), axis=-1)[0]
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

print("\n Example Generated Text:")
print(generate_text("to be or not to", next_words=15))


[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


 Data ready: X=(4000, 4001), y=(4000,), vocab=4797
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


57/57 ━━━━━━━━━━━━━━━━━━━━ 730s 13s/step - accuracy: 0.0330 - loss: 8.0047 - val_accuracy: 0.0375 - val_loss: 6.5783
Epoch 2/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 732s 13s/step - accuracy: 0.0328 - loss: 6.0446 - val_accuracy: 0.0325 - val_loss: 6.4830
Epoch 3/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 745s 13s/step - accuracy: 0.0397 - loss: 5.8668 - val_accuracy: 0.0375 - val_loss: 6.5608
Epoch 4/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 712s 12s/step - accuracy: 0.0323 - loss: 5.8606 - val_accuracy: 0.0325 - val_loss: 6.5761
Epoch 5/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 756s 13s/step - accuracy: 0.0357 - loss: 5.7829 - val_accuracy: 0.0375 - val_loss: 6.5988
Epoch 6/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 713s 13s/step - accuracy: 0.0328 - loss: 5.7325 - val_accuracy: 0.0325 - val_loss: 6.6570
Epoch 7/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 715s 13s/step - accuracy: 0.0353 - loss: 5.6925 - val_accuracy: 0.0325 - val_loss: 6.6781
Epoch 8/10
57/57 ━━━━━━━━━━━━━━━━━━━━ 749s 13s/step - accuracy: 0.0405 - loss: 5.6631 - val_accuracy: 0.0525 - val_